In [1]:
import sqlite3
from sentence_transformers import SentenceTransformer, util

/Users/alexandre/PycharmProjects/tea/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
db = sqlite3.connect('../dataMining/tea.db')
cursor = db.cursor()

In [9]:
description = cursor.execute("SELECT description FROM Tea;").fetchall()
description

[('Elevate your senses with this blend crafted by our herbalists to support joy and inner harmony. With the uplifting herbal power of damiana and rose, and the unique addition of butterfly pea flower to inspire a sense of wonder, this tea supports a happy mood and an open heart. Enjoy this mildly tart & sweet floral tea whenever you need a little extra positivity to brighten your day.',),
 ('This blend of adaptogens and calming nervines helps build your resilience to occasional stress. Ashwagandha is combined with shatavari, harnessing the power of adaptogenic herbs to support a balanced mood and help restore the spirit. Blended with rose flower to calm the nerves and featuring sunny honeybush, this floral and mildly sweet tea is the ideal go-to everyday soother.',),
 ('Lemon Balm tea calms the nervous system and supports digestion. It will brighten your day with subtle citrus notes.',),
 ('Organic Nighty Night Extra tea infuses the power of valerian root, a gentle, time-tested herbal 

In [10]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11759.30it/s]


In [23]:
def normalize(data: list):
    description_form = []
    for descript in data:
        description_form.append(descript[0])
    return description_form

def normalize_ex(data: list):
    description_form = []
    for descript in data:
        description_form.extend(descript[0])
    return description_form

In [16]:
corpus = model.encode(normalize(description), device="mps", normalize_embeddings=True)
corpus

array([[-0.07631826, -0.01986498,  0.09798845, ...,  0.01078916,
         0.08592612, -0.00813074],
       [-0.01053185, -0.02590794, -0.02039999, ...,  0.03636362,
         0.01254994, -0.011708  ],
       [ 0.02207725, -0.03888993,  0.01695614, ...,  0.10636193,
        -0.06408373, -0.04241326],
       ...,
       [-0.02443083, -0.06758369, -0.01992207, ...,  0.03106632,
         0.20880958, -0.03774684],
       [-0.08645435, -0.0443989 , -0.02338292, ..., -0.01367737,
         0.12757313, -0.04400383],
       [-0.05190955, -0.04909388, -0.00223628, ..., -0.03679322,
        -0.05501366,  0.00408322]], shape=(51, 384), dtype=float32)

In [20]:
tea_ids = normalize(cursor.execute("SELECT id from Tea;").fetchall())
tea_ids

[10,
 28,
 11,
 21,
 13,
 42,
 41,
 32,
 24,
 30,
 45,
 6,
 4,
 25,
 40,
 9,
 8,
 5,
 15,
 20,
 51,
 12,
 17,
 36,
 16,
 19,
 3,
 22,
 29,
 47,
 38,
 14,
 46,
 7,
 44,
 34,
 48,
 33,
 2,
 1,
 43,
 37,
 26,
 23,
 27,
 39,
 35,
 18,
 49,
 31,
 50]

In [27]:
formula_tea = []
for id in tea_ids:
    formula_tea.append(" ".join(normalize(cursor.execute("SELECT herb FROM Ingredients WHERE tea = ?", (id,)).fetchall())))
formula_tea

['Eucalyptus Ginger Licorice Peppermint',
 'Licorice Linden Meadowsweet Passionflower',
 'Catnip Chamomile Lavender Licorice Passionflower Peppermint Spearmint',
 'Hibiscus Licorice Nettle Schisandra Berry',
 'Chamomile Echinacea Elder Ginger Peppermint',
 'Echinacea Lemongrass Spearmint',
 'Ginger Lemon myrtle',
 'Tea Plant',
 'Elder Ginger Licorice Peppermint',
 'Echinacea Elder Hibiscus Licorice',
 'Echinacea Ginger Lemon Balm Licorice',
 'Nettle',
 'Lemon Balm Licorice Passionflower Peppermint Valerian',
 'Catnip Chamomile Hops Linden Passionflower Spearmint',
 'Chamomile Ginger',
 'Hawthorn Hibiscus',
 'Chamomile Lavender Lemon Balm',
 'Dandelion',
 'Hibiscus Lemongrass',
 'Raspberry Leaf',
 'Ginger Meadowsweet Turmeric',
 'Peppermint',
 'Dandelion',
 'Ginger Hibiscus Lemongrass',
 'Ginger',
 'Chamomile',
 'Lemon Balm',
 'Fennel',
 'Licorice',
 'Astragalus Eleuthero Hawthorn Reishi',
 'Eleuthero Ginseng Hibiscus Rose Hips Spearmint Wild Apple',
 'Spearmint',
 'Skullcap Spearmint',

In [28]:
corpus_formula = model.encode(formula_tea, device="mps", normalize_embeddings=True)
corpus_formula

array([[-0.05340098, -0.02595015, -0.0229991 , ..., -0.08335655,
         0.03372296,  0.16429996],
       [ 0.05579627, -0.02689048,  0.03310459, ..., -0.02803234,
         0.01105991,  0.02053915],
       [-0.04886077, -0.06339472,  0.02513671, ..., -0.00307238,
         0.01243658,  0.08584227],
       ...,
       [-0.04266636,  0.03196707, -0.03173059, ..., -0.04852212,
         0.10066882,  0.11310881],
       [-0.06436867,  0.00353007, -0.07464942, ..., -0.00955307,
         0.02078425,  0.05884367],
       [-0.05353867,  0.02723732, -0.0490977 , ..., -0.03345013,
         0.0836408 ,  0.06451922]], shape=(51, 384), dtype=float32)

In [34]:
placeholders = ",".join("?" for _ in tea_ids)

names = normalize(cursor.execute(f"SELECT name FROM Tea WHERE id IN ({placeholders});", tea_ids).fetchall())
names

['Rosy Mood™ Tea',
 'Stress Ease® Calm Tea',
 'Lemon Balm Tea',
 'Nighty Night Extra® Tea',
 'Dandelion Leaf & Root Tea',
 'Nettle Leaf Tea',
 'Throat Coat® Tea',
 'Chamomile & Lavender Tea',
 'Hawthorn & Hibiscus Tea',
 'Breathe Easy® Tea',
 'Cup of Calm® Tea',
 'Peppermint Tea',
 'Echinacea Plus® Elderberry Tea',
 'Spearmint Tea',
 'Hibiscus Tea',
 'Ginger Tea',
 'Roasted Dandelion Root Tea',
 'Smooth Move® Tea',
 'Chamomile Tea',
 'Raspberry Leaf Tea',
 'EveryDay Detox® Hibiscus & Schisandra Berry Tea',
 'Fennel Tea',
 'EveryDay Detox® Lemon Tea',
 'Herbal Cold Care™ Tea',
 'Nighty Night® Tea',
 'Healthy Cycle® Tea',
 'Belly Comfort® Peppermint Tea',
 'Cold Care P.M.® Tea',
 'Licorice Root Tea',
 'Immune Zoom® Elderberry Echinacea Tea',
 'Green Tea Ginger',
 'Green Tea Matcha',
 'Triple Mint Daily Probiotic Tea',
 'Throat Coat® Lemon Echinacea Tea',
 'Gas Relief™ Tea',
 'Lemon Ginger Tea',
 'Green Tea Lemongrass',
 'Stress Ease® Focus Tea',
 'Green Tea Peppermint',
 'Ginger & Chamom

In [35]:
names_formula = model.encode(names, device="mps", normalize_embeddings=True)
names_formula

array([[-0.04751815, -0.0072734 ,  0.00543021, ...,  0.02356526,
         0.05792335, -0.01544003],
       [-0.02451778, -0.01097845,  0.00675668, ...,  0.04751345,
         0.02912137,  0.03757538],
       [-0.05983679,  0.01549164, -0.03567895, ...,  0.00772358,
        -0.02218311,  0.0219468 ],
       ...,
       [-0.03165069, -0.06223337,  0.00282547, ..., -0.02059575,
         0.1845588 , -0.05773344],
       [-0.08377947, -0.01478989, -0.02952757, ..., -0.04739074,
         0.15709005, -0.00164981],
       [-0.04431125, -0.04279358, -0.03883456, ..., -0.05643068,
        -0.05572754,  0.02182545]], shape=(51, 384), dtype=float32)

In [36]:
import faiss

dimension = 384

name_index = faiss.IndexFlatIP(dimension)
description_index = faiss.IndexFlatIP(dimension)

In [37]:
name_index.add(names_formula)
description_index.add(corpus)

In [39]:
faiss.write_index(name_index, "tea_names.index")
faiss.write_index(description_index, "tea_descriptions.index")

In [68]:
placeholders = ",".join("?" for _ in [1,2,3,4,5])

distance, idx = name_index.search(model.encode(["Rosy Mood™ Tea"], normalize_embeddings=True), 5)
d = cursor.execute(f"SELECT * FROM Tea WHERE id IN ({placeholders});", idx[0].tolist()).fetchall()
d

[(1,
  'Rosy Mood™ Tea',
  'Elevate your senses with this blend crafted by our herbalists to support joy and inner harmony. With the uplifting herbal power of damiana and rose, and the unique addition of butterfly pea flower to inspire a sense of wonder, this tea supports a happy mood and an open heart. Enjoy this mildly tart & sweet floral tea whenever you need a little extra positivity to brighten your day.',
  0,
  0,
  'https://www.traditionalmedicinals.com/products/rosy-mood-tea',
  'https://cdn.shopify.com/s/files/1/0506/7037/0997/files/tdm-008503-rosy-mood_1.png?v=1761596443',
  7.49,
  'Do not use this product if you are allergic to any of the listed ingredients. Consult a healthcare provider before use if you are breastfeeding , or if you are taking cardiac glycosides such as digitalis/digoxin, or blood pressure medication. Not recommended for use during pregnancy or by children under 12 years of age . Do not exceed recommended dosage .',
  'Pour 8 oz. freshly boiled water ove